# Condition Shift Baseline Notebook

이 노트북은 `condition_shift_baseline` 실험의 thin orchestrator다.

- Colab/서버에서 Git checkout 상태를 확인한다.
- 필요하면 prepared dataset bootstrap script를 호출한다.
- versioned Python runner를 실행한다.
- `summary.json`과 `log.txt`를 읽어 표와 간단 시각화를 보여준다.

실험 핵심 로직은 노트북에 두지 않고, notebook이 호출하는 `.py`는 repo에 versioned 상태로 유지한다.


## 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
repo_candidates = [cwd, *cwd.parents, Path('/content/ReGraM'), Path('/content/ReGRaM')]
REPO_ROOT = next((p.resolve() for p in repo_candidates if p.exists() and p.name in {'ReGraM', 'ReGRaM'}), cwd)
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Git Checkout

Colab에서는 먼저 repo를 Git으로 clone 또는 pull 해서 notebook이 사용할 `.py`를 최신 상태로 맞춘다.


In [ ]:
REPO_URL = 'https://github.com/outSeop/ReGraM.git'
REPO_DIR = Path('/content/ReGraM')
git_cmd = (
    f'if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" pull --ff-only; '
    f'else git clone "{REPO_URL}" "{REPO_DIR}"; fi'
)
print(git_cmd)
subprocess.run(['bash', '-lc', git_cmd], check=True)

REPO_ROOT = REPO_DIR.resolve()
EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
REPORT_ROOT = EXP_ROOT / 'reports'
NOTEBOOK_ROOT = EXP_ROOT / 'notebook'
MANIFEST_ROOT_CANDIDATES = [REPO_ROOT / 'manifests', EXP_ROOT / 'manifests']

print('updated REPO_ROOT =', REPO_ROOT)
print('updated EXP_ROOT =', EXP_ROOT)


## Dataset load

In [ ]:
from pathlib import Path
import subprocess

drive_tar = Path("/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz")
runtime_tar = Path("/content/mvtec_loco_anomaly_detection.tar.gz")
runtime_row = Path("/content/ReGraM/data/row")
runtime_root = runtime_row / "mvtec_loco_anomaly_detection"

print("drive_tar exists:", drive_tar.exists())
print("runtime_root exists:", runtime_root.exists())

runtime_row.mkdir(parents=True, exist_ok=True)

if not runtime_root.exists():
    if not runtime_tar.exists():
        subprocess.run(["cp", str(drive_tar), str(runtime_tar)], check=True)
    subprocess.run(
        ["tar", "-xf", str(runtime_tar), "-C", str(runtime_row)],
        check=True,
    )

print("done")
print(runtime_root.exists(), runtime_root)


## Optional Dataset Bootstrap

Git checkout 이후 prepared dataset이나 작은 reports 자산이 필요하면 아래처럼 별도 Python bootstrap script를 호출한다.
코드 동기화는 bootstrap이 아니라 Git이 담당한다.


In [ ]:
bootstrap_cmd = [
    sys.executable,
    str(EXP_ROOT / 'colab' / 'bootstrap_runtime.py'),
    '--dry-run',
]
print(' '.join(bootstrap_cmd))
subprocess.run(bootstrap_cmd, cwd=REPO_ROOT, check=True)


## Runner Orchestration

노트북은 어떤 runner를 어떤 인자로 호출할지 만 관리한다.


## Runner Config

manifest와 category를 먼저 정하고, runner와 summary viewer가 같은 설정을 공유하게 둔다.


In [ ]:
CATEGORY = 'breakfast_box'
MANIFEST_NAME = 'query_motion_blur.jsonl'

manifest_path = next(
    (root / MANIFEST_NAME for root in MANIFEST_ROOT_CANDIDATES if (root / MANIFEST_NAME).exists()),
    None,
)
if manifest_path is None:
    searched = [str(root / MANIFEST_NAME) for root in MANIFEST_ROOT_CANDIDATES]
    raise FileNotFoundError(f'Manifest not found. searched={searched}')

manifest_stem = manifest_path.stem
augmentation_suffix = manifest_stem.removeprefix('query_') if manifest_stem.startswith('query_') else manifest_stem
summary_path = REPORT_ROOT / 'patchcore_manifest_shift' / f'{CATEGORY}_{augmentation_suffix}.json'
runner_cmd = [
    sys.executable,
    str(EXP_ROOT / 'src' / 'core' / 'run_patchcore_manifest_shift.py'),
    '--category', CATEGORY,
    '--manifest', str(manifest_path),
]

print('manifest_path =', manifest_path)
print('summary_path =', summary_path)
print('runner_cmd =', ' '.join(runner_cmd))


In [ ]:
!mkdir -p /content/ReGraM/external
!test -d /content/ReGraM/external/patchcore-inspection.clean/.git || git clone https://github.com/amazon-science/patchcore-inspection.git /content/ReGraM/external/patchcore-inspection.clean
!pip install faiss-cpu timm


In [ ]:
result = subprocess.run(
    runner_cmd,
    cwd=REPO_ROOT,
    text=True,
    capture_output=True,
)

print('returncode:', result.returncode)
print('stdout:
', result.stdout)
print('stderr:
', result.stderr)


## Summary Viewer

runner가 남긴 `summary.json`과 `log.txt`를 기준으로만 결과를 본다.


In [ ]:
if not summary_path.exists():
    display(Markdown(f'`summary.json` not found: `{summary_path}`'))
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    display(Markdown(f"### {summary['baseline']} / {summary['class_name']} / {summary['eval_type']}"))
    display(pd.DataFrame([summary['metrics']]))
    display(pd.DataFrame([summary['paths']]))
    payload = summary.get('payload', {})
    aug_rows = []
    for aug_type, severity_map in payload.get('augmentations', {}).items():
        for severity, item in severity_map.items():
            aug_rows.append({
                'augmentation_type': aug_type,
                'severity': severity,
                'mean': item.get('mean'),
                'fpr_over_clean_max': item.get('fpr_over_clean_max'),
                'mean_score_shift': item.get('mean_score_shift'),
            })
    if aug_rows:
        display(pd.DataFrame(aug_rows))
    log_path = Path(summary['paths']['log_path'])
    if log_path.exists():
        print(log_path.read_text(encoding='utf-8'))
